
# Plan: TIFF Pathology De-identification Pipeline
## Architecture Plan — TIFF→ De-identified TIFF

Extends [databricks-industry-solutions/pixels](https://github.com/databricks-industry-solutions/pixels) to treat  `.tiff` as a first-class format alongside DICOM.

---

## 1. Confirmed Inputs & Outputs

| Item | Value |
|---|---|
| Input SVS path | `/Volumes/hls_pathology/orthanc_demo/raw_images/Aperio/` |
| Demo scale | ~14 files; architecture targets **10 million** |
| Output catalog / schema | `douglas_moore.pathology` (to be created) |
| TIFF output volume | `/Volumes/douglas_moore/pathology/tiff_deidentified/` |
| Label images volume | `/Volumes/douglas_moore/pathology/label_images/` |
| VLM endpoint | `databricks-llama-4-maverick` (config param) |
| Redaction method | Black rectangle fill |
| Source SVS | **Read-only** — never modified |

---

## 2. Delta Table Schema

> **No new DDL is needed for SVS ingest.** The `object_catalog` table is used exactly as defined in the base `CREATE_OBJECT_CATALOG.sql` — no columns are added or altered. All SVS-specific metadata (dimensions, pyramid levels, sub-image presence, PHI tag classification) is serialised into the existing `meta VARIANT` column and accessed via VARIANT path syntax. `SVSCatalog.init_tables()` calls `super().init_tables()` which runs the unmodified base DDL against the `douglas_moore.pathology` schema. The only new DDL is the `_redaction` table.

### `douglas_moore.pathology.object_catalog` *(base DDL, unchanged)*
One row per SVS file. Populated by `SVSCatalog.catalog()` + `SVSMetaExtractor`.

| Column | Type | Notes |
|---|---|---|
| `path` | STRING NOT NULL | Cloud storage path |
| `modificationTime` | TIMESTAMP NOT NULL | |
| `length` | BIGINT NOT NULL | File size bytes |
| `original_path` | STRING | |
| `relative_path` | STRING | |
| `local_path` | STRING NOT NULL | Worker-accessible path — **`inputCol` for all Transformers** |
| `extension` | STRING | `"svs"` |
| `file_type` | STRING | |
| `path_tags` | ARRAY\<STRING\> | From `TagExtractor` |
| `is_anon` | BOOLEAN | |
| `meta` | **VARIANT** | All OpenSlide properties + SVS-specific fields serialised together. Query with `meta:aperio.Date::string`, `meta:width::int`, `meta:has_label_image::boolean` |

**SVS fields stored inside `meta VARIANT`** (no schema change required):
- `meta:width::int`, `meta:height::int` — level-0 pixel dimensions
- `meta:level_count::int` — pyramid depth
- `meta:has_label_image::boolean`, `meta:has_macro_image::boolean`
- `meta:phi_tag_report` — array of `{tag, value, classification}` structs
- All raw OpenSlide properties (e.g. `meta:"aperio.AppMag"::string`)

### `douglas_moore.pathology.object_catalog_redaction` *(unified DICOM + SVS DDL)*
One row per redaction job, for any format. Created by `CREATE_SVS_CATALOG.sql`.

Three DICOM columns are renamed to remove format-specific semantics; new columns cover VLM detection results and SVS artefacts. All new and renamed columns are nullable for backward compatibility.

**Format-agnostic identifiers**

| Column | Type | Notes |
|---|---|---|
| `redaction_id` | STRING NOT NULL | UUID per job |
| `path` | STRING | FK → `object_catalog.path` *(new — not in DICOM original)* |
| `extension` | STRING | Discriminator: `dcm`, `svs`, `czi` … *(new)* |

**DICOM identifiers** *(NULL for SVS)*

| Column | Type | Notes |
|---|---|---|
| `study_instance_uid` | STRING | DICOM Study UID |
| `series_instance_uid` | STRING | DICOM Series UID |
| `modality` | STRING | DICOM modality, or `WSI` for SVS |
| `new_series_instance_uid` | STRING | New UID for redacted DICOM series |

**Redaction configuration** *(both formats)*

| Column | Type | Change from DICOM original |
|---|---|---|
| `redaction_config` | VARIANT | **Renamed** from `redaction_json` |
| `metadata_redactions_count` | INT | **Renamed** from `global_redactions_count` |
| `pixel_redactions_count` | INT | **Renamed** from `frame_specific_redactions_count` |
| `total_redaction_areas` | INT | Unchanged |
| `phi_tags_redacted` | ARRAY\<STRING\> | Tag names scrubbed *(new)* |

**VLM PHI detection results** *(new — both formats)*

| Column | Type | Notes |
|---|---|---|
| `has_phi` | BOOLEAN | VLM verdict |
| `phi_elements` | VARIANT | Detected regions: type, value\_hint, bbox |
| `vlm_raw_response` | STRING | Raw model output |
| `model_endpoint` | STRING | Endpoint name |

**Output paths**

| Column | Type | Notes |
|---|---|---|
| `output_file_paths` | ARRAY\<STRING\> | DICOM: one `.dcm` per slice. SVS: single TIFF at index 0 |
| `label_image_path` | STRING | De-identified label PNG *(SVS only, NULL for DICOM)* |
| `macro_image_path` | STRING | De-identified macro PNG *(SVS only, NULL for DICOM)* |

**Processing status & audit** *(unchanged from DICOM original)*
`status`, `error_messages`, `insert_timestamp`, `update_timestamp`, `processing_start_timestamp`, `processing_end_timestamp`, `processing_duration_seconds`, `created_by`, `export_timestamp`

---


## 3. Python Package: `dbx.pixels.svs`

> **Pattern source: actual repo code** — All transformers extend `pyspark.ml.pipeline.Transformer` (Spark ML, not a custom base). The main entry-point for file cataloguing is the `Catalog` class, not a `Processor`. There is no `Processor` in the repo. The CZI extractor (`src/dbx/pixels/czi/`) is a stub — SVS is genuinely the first completed non-DICOM format extension.

Namespace-package extension of `dbx-pixels`. Created as workspace files under `svs-pixels/src/`, installed via `%pip install -e ./src`.

```
svs-pixels/
├── src/
│   └── dbx/
│       └── pixels/
│           └── svs/
│               ├── __init__.py          ← exports SVSCatalog, SVSMetaExtractor, SVSTiffWriter, SVSPhiPipeline
│               ├── catalog.py           ← SVSCatalog(Catalog)
│               ├── svs_meta_extractor.py← SVSMetaExtractor(Transformer)
│               ├── svs_tiff_writer.py   ← SVSTiffWriter(Transformer)
│               ├── phi_tags.py          ← PHI classification lookup dict
│               └── deidentify.py        ← pixel redaction helpers
│               └── resources/sql/
│                   └── CREATE_SVS_CATALOG.sql  ← creates object_catalog_redaction only
├── pyproject.toml
└── (this notebook)
```

### `SVSCatalog` (extends `Catalog`)
- Calls `super().__init__(spark, table, volume)` — reuses all existing table management, volume, and Auto Loader infrastructure
- `catalog(path, pattern="*.svs", ...)` → delegates to `Catalog.catalog()` with SVS glob pattern; callers never need to pass `pattern`
- `init_tables()` → calls `super().init_tables()` (creates `object_catalog` via unmodified base DDL), then executes one SVS-specific file — `resources/sql/CREATE_SVS_CATALOG.sql` — which creates only the `object_catalog_redaction` table with SVS-specific columns

### `SVSMetaExtractor` (extends `pyspark.ml.pipeline.Transformer`)
Mirrors `DicomMetaExtractor`: uses `mapInPandas` with `ThreadPoolExecutor` for concurrent I/O (optimal for network-bound OpenSlide reads).

```python
class SVSMetaExtractor(Transformer):
    def __init__(self, catalog, inputCol="local_path", outputCol="meta",
                 maxWorkers=32, useVariant=True): ...

    def _transform(self, df):   # Spark ML Transformer contract
        # mapInPandas with ThreadPoolExecutor — same pattern as DicomMetaExtractor
        ...
```

**Single output column written to `object_catalog`:**

| Column | Spark type | Notes |
|---|---|---|
| `meta` | `VARIANT` | OpenSlide properties dict merged with derived fields (`width`, `height`, `level_count`, `has_label_image`, `has_macro_image`, `phi_tag_report`) into one JSON object, then `parse_json()`'d into VARIANT |

All SVS-specific fields are embedded inside `meta` before serialisation — no extra top-level columns are written, no `ALTER TABLE` or `mergeSchema` required. VARIANT path syntax handles all downstream access: `meta:width::int`, `meta:phi_tag_report[0].classification::string`, etc.

### `SVSTiffWriter` (extends `pyspark.ml.pipeline.Transformer`)
Converts SVS → de-identified pyramidal BigTIFF. Wraps the write logic in `_transform(df)` operating on the output of `SVSMetaExtractor`.

### `SVSPhiPipeline` (extends `pyspark.ml.Pipeline`)
Composed pipeline, mirrors `DicomPhiPipeline`:
```
Stage 1: SVSMetaExtractor   → adds meta VARIANT + phi_tag_report
Stage 2: SVSVlmPhiDetector  → adds phi_elements (VLM bboxes on label/macro)
Stage 3: SVSFilterTransformer → nullifies rows with no PHI detected
Stage 4: SVSTiffWriter       → writes de-identified BigTIFF + audit log
```

---





## 4. PHI Tag Classification (`phi_tags.py`)

Research-hardcoded lookup covering all standard OpenSlide / Aperio TIFF properties:

| Classification | Example Tags |
|---|---|
| `PHI` | `aperio.Patient`, `aperio.PatientID`, `aperio.DOB`, `aperio.MRN`, `aperio.AccessionNumber`, `aperio.ClinicID`, `aperio.ClinicalTrialID`, `aperio.Procedure`, `tiff.ImageDescription` (contains patient name in Aperio format) |
| `QUESTIONABLE` | `aperio.Date`, `aperio.Time`, `aperio.Clinic`, `aperio.Pathologist`, `tiff.Artist`, `tiff.Copyright`, `aperio.Title`, `aperio.Filename`, `aperio.User`, `aperio.ImageID` |
| `NOT_PHI` | `aperio.AppMag`, `aperio.MPP`, `aperio.ScanScope ID`, `openslide.level-count`, `openslide.mpp-x`, `openslide.mpp-y`, `openslide.objective-power`, `tiff.Make`, `tiff.Model`, `tiff.Software`, `openslide.vendor`, all `openslide.level[N].*` pyramid geometry tags |

Function `classify_tags(properties: dict) → list[dict]` iterates all OpenSlide properties and returns the structured report stored in `phi_tag_report`.

---

## 5. VLM PHI Detection via `ai_query()`

### What is Inspected
Aperio SVS files embed three sub-images accessible via `slide.associated_images` — **confirmed from real files**:

| Sub-image | Dims (CMU-1) | Mode | PHI Risk | Action |
|---|---|---|---|---|
| `label` | 387×463 | RGBA | **HIGH** — physical paper label with patient name, barcode, accession | VLM analysis + black-box redaction |
| `macro` | 1280×431 | RGBA | **MEDIUM** — full-slide photo; label region visible at right edge | VLM analysis + label region redaction |
| `thumbnail` | 1024×732 | RGBA | LOW — auto-generated tissue preview | Excluded from output |

The tissue scan (`level 0`: 46000×32914) is in a **completely separate coordinate space** from the label/macro sub-images. PHI in the tissue scan itself is rare but possible (e.g., handwriting on the glass).

The label image is the **primary** VLM target. Macro is secondary.

### Pipeline
1. `SVSTransformer.extract_embedded_images()` saves label and macro PNGs to `/Volumes/douglas_moore/pathology/label_images/` using the naming convention `{slide_name}_label.png` / `{slide_name}_macro.png`
2. Run `ai_query()` directly via `READ_FILES()` on the volume — **no binary column staging needed**:

```sql
INSERT INTO douglas_moore.pathology.phi_pixel_audit
SELECT
  m.path,
  f._metadata.file_path                          AS label_image_path,
  ai_query(
    'databricks-llama-4-maverick',
    'You are a medical PHI detection system analyzing a pathology slide label.
     Return ONLY valid JSON:
     {"has_phi": bool,
      "phi_elements": [{"type": "name|dob|mrn|accession|barcode|other",
                        "value_hint": "first 3 chars only",
                        "bbox": {"x": int, "y": int, "w": int, "h": int}}]}
     Bounding box coordinates are in the label image pixel space (origin top-left).',
    files => f.content
  )                                              AS vlm_raw_response,
  'databricks-llama-4-maverick'                  AS model_endpoint,
  current_timestamp()                            AS inferred_at
FROM read_files(
  '/Volumes/douglas_moore/pathology/label_images/',
  format => 'binaryFile',
  fileNamePattern => '*_label.png'
) f
JOIN douglas_moore.pathology.svs_metadata m
  ON m.filename = regexp_replace(f._metadata.file_name, '_label\.png
```
---



## 6. De-identification & TIFF Output (`deidentify.py` / `SVSTiffWriter`)

> **Pattern source: actual repo code** — `DicomPhiPipeline` uses a two-stage approach: (1) `VLMPhiDetector` returns a **pipe-separated list of PHI text strings** (`'John Smith'|'04-31-1954'`), NOT bboxes. (2) `OcrRedactor` then runs EasyOCR on the image to locate those strings and draw black rectangles. The VLM provides *what* is PHI; OCR provides *where*. SVS uses this same two-stage approach.

### Revised De-identification Algorithm

#### Stage 1 — VLM PHI Detection (`SVSVlmPhiDetector`, extends `Transformer`)
- Submit label/macro PNGs via `ai_query()` with `files => content`
- Prompt returns a **pipe-separated list of PHI entity strings** (consistent with the SA pattern)
- Optionally also request bboxes via `responseFormat => json_schema` (SVS-specific addition for direct redaction without a second OCR pass)

#### Stage 2 — Pixel Redaction (`SVSTiffWriter._transform(df)`)
1. Open SVS with `openslide.OpenSlide(local_path)`
2. Read level-0 in 4096×4096 tiles using `slide.read_region()`
3. If bbox-only mode: draw filled black `PIL.ImageDraw.rectangle` over each detected region in label/macro
4. If text-only mode: run EasyOCR on label image to locate the strings from the VLM response, then black-out matching text (mirrors `OcrRedactor`)
5. Scrub PHI tags in `tiff.ImageDescription` using `phi_tags.scrub_image_description()`
6. Write pyramidal BigTIFF using `tifffile.TiffWriter(bigtiff=True)` with `subifds=level_count-1` and 256×256 JPEG tiles
7. Return `(tiff_output_path, phi_tags_redacted_list, pixel_regions_count)` for the audit row

### VLM Implementation: Two Approaches

| Approach | Used by | Library | Scale |
|---|---|---|---|
| OpenAI SDK + base64 | `VLMPhiExtractor` in pixels SA | `openai` Python SDK, `pandas_udf` | Single-node / moderate |
| `ai_query()` + `files => content` | **Our SVS pipeline** | Databricks SQL / Spark SQL | 10M images, serverless SQL |

For the demo scale, both work. For 10M, `ai_query()` via SQL is the correct choice — it delegates throughput management to the Databricks SQL engine.



## 7. Notebook Cell Structure

| Cell | Purpose |
|---|---|
| **Cell 1** | This plan (markdown) |
| **Cell 2** | `%pip install dbx-pixels openslide-python openslide-bin tifffile Pillow easyocr` + `%pip install -e ./src` |
| **Cell 3** | Configuration: paths, catalog, schema, volume names, model endpoint |
| **Cell 4** | Storage bootstrap: `SVSCatalog(spark, ...).init_tables()` — `super().init_tables()` creates `object_catalog` (base DDL, unchanged); `CREATE_SVS_CATALOG.sql` creates `object_catalog_redaction` (SVS-specific columns only) |
| **Cell 5** | File discovery: `SVSCatalog.catalog(INPUT_PATH)` → writes `object_catalog` (pattern defaults to `"*.svs"`) |
| **Cell 6** | Metadata extraction: `SVSMetaExtractor(catalog)._transform(files_df)` → populates `meta VARIANT` (OpenSlide properties + derived SVS fields merged into one JSON object) |
| **Cell 7** | PHI tag report: SQL on `object_catalog` using VARIANT path syntax (`meta:aperio.Date::string`, `meta:phi_tag_report`) |
| **Cell 8** | Label/macro image extraction → `/Volumes/.../label_images/` |
| **Cell 9** | VLM inference: `ai_query()` SQL → `object_catalog_redaction` |
| **Cell 10** | De-identified TIFF write: `SVSTiffWriter._transform(df)` → BigTIFFs to volume |
| **Cell 11** | Audit summary: join `object_catalog` + `object_catalog_redaction`, show statistics |

---


## 8. Scale Architecture Notes (10M Images)

| Concern | Demo Approach | 10M Approach |
|---|---|---|
| File discovery | `dbutils.fs.ls` recursive | Auto Loader on the volume path |
| Metadata extraction | `SVSMetaExtractor` via `mapInPandas` + `ThreadPoolExecutor` | Same — already distributed |
| Label image storage | Written to volume as files | Stored as `BINARY` in Delta table (eliminates extra volume I/O) |
| VLM inference | Single SQL batch `ai_query()` | Incremental: `WHERE vlm_status='PENDING'` in a scheduled Lakeflow Job |
| TIFF conversion | `write_deidentified_tiff_udf` Spark UDF | Same — Photon-accelerated UDF dispatch |
| Checkpointing | `vlm_status` column | Same + Delta transaction log for idempotency |
| Cost control | Serverless interactive | SQL Serverless warehouse + compute-optimized clusters for UDF stages |

---

## 9. Confirmed Findings & Resolved Design Decisions

All four open questions are now resolved from direct inspection of the actual Aperio CMU-1.svs files.

### Q1 — Bounding box coordinate space ✅ RESOLVED

The label sub-image is **387×463 RGBA** — completely independent from the tissue scan (46000×32914). The two coordinate spaces share no relationship.

**Decision:** Save and submit the label image to the VLM at **native resolution (no resizing)**. All VLM bboxes are in label-image pixel space (`0,0` = top-left). The tissue TIFF does **not** embed the label — it is automatically excluded when only the main pyramid is read. The de-identified label PNG (black rectangles applied) is written to the `label_images` volume as the audit artefact.

**Macro image clarification:** The macro (1280×431) shows both tissue and the physical label (at one end). It is submitted to the VLM separately; its bboxes drive black-rectangle redaction of the label area in the macro output PNG.

---

### Q2 — `tiff.ImageDescription` scrubbing ✅ RESOLVED

Exact format confirmed from the real file:
```
Aperio Image Library v10.0.51\r\n46920x33014 [0,100 46000x32914] (256x256) JPEG/RGB Q=30
  |AppMag = 20|StripeWidth = 2040|ScanScope ID = CPAPERIOCS|Filename = CMU-1
  |Date = 12/29/09|Time = 09:59:15|User = b414003d-...|ImageID = 1004486|...
```

**Structure:** `{header_line}|key = val|key = val|...`  Header is technical-only — preserve as-is.

**PHI classification of actual keys:**

| Key | Classification | Notes |
|---|---|---|
| `Date`, `Time` | PHI | HIPAA date/time of service |
| `User` | QUESTIONABLE | GUID in demo; operator name in clinical use |
| `Filename` | QUESTIONABLE | May encode patient name or MRN |
| `ImageID` | QUESTIONABLE | Could be accession number |
| `ScanScope ID`, `AppMag`, `StripeWidth`, `Parmset`, `MPP`, all geometry/calibration, `Filtered`, `ICC Profile` | NOT_PHI | Pure scanner parameters |

Clinical files may also contain: `Patient`, `DOB`, `MRN`, `AccessionNumber`, `Clinic`, `Pathologist`, `Procedure`, `Diagnosis`, `Id` — all PHI.

**Scrubbing algorithm:**
1. `header, *kvs = image_desc.split('|')`
2. For each `kv`: `k, v = kv.split(' = ', 1)` — rebuild as `k = REDACTED` if `k.strip()` ∈ PHI/QUESTIONABLE set
3. Rejoin: `'|'.join([header] + rebuilt_kvs)`
4. Apply identical scrub to `openslide.comment` (same content) when writing TIFF metadata

---

### Q3 — Macro image redaction ✅ RESOLVED

Macro (1280×431) shows the full physical slide including the affixed label. **Decision:** Include macro in the primary VLM pipeline alongside the label (not a follow-on phase). Naming: `{name}_label.png` / `{name}_macro.png`. Both de-identified PNGs go to the `label_images` volume.

---

### Q4 — Pyramidal TIFF output ✅ RESOLVED

Flat TIFF is not viable for pathology — QuPath, OMERO, and DIGIPATH all require pyramidal. The source SVS has 3 levels with 256×256 tiles; match this in output.

**Decision:** Write pyramidal **BigTIFF** via `tifffile`:
- `bigtiff=True` — mandatory (CMU-1 level-0 ~7.4 GB uncompressed, exceeds 4 GB TIFF limit)
- `tile=(256, 256)` — matches native Aperio tile size
- `compression='jpeg'` at quality 80; swap to `'lzw'` if lossless required
- `subifds=level_count - 1` — sub-IFDs are the QuPath/libvips-compatible pyramid convention
- Pyramid levels: 2× progressive downsampling with `PIL.Image.LANCZOS`

```python
with tifffile.TiffWriter(output_path, bigtiff=True) as tif:
    opts = dict(tile=(256, 256), compression='jpeg',
                compressionargs={'level': 80}, photometric='rgb', metadata=None)
    tif.write(level_0_rgb,           subifds=level_count - 1, **opts)  # main IFD
    for lvl in range(1, level_count):
        tif.write(level_arrays[lvl], subfiletype=1, **opts)             # sub-IFDs
```

OME-TIFF (`ome=True`) only if OMERO is a confirmed downstream consumer.

---


## 10. More...

### Additional: `openslide-bin` Required

`openslide-python` alone fails at import on Databricks Serverless:
```
ModuleNotFoundError: Couldn't locate OpenSlide shared library. Try pip install openslide-bin.
```
**Cell 2 must install:** `openslide-python openslide-bin` (the `openslide-bin` wheel bundles `libopenslide.so` for environments without system package access).


> **Note**: `files => content` is the correct `ai_query()` API for binary image inputs — it passes the PNG bytes directly to the model without base64 encoding. Only JPEG and PNG inputs are supported.

### Scale to 10M Images
- `vlm_status` column acts as a watermark: `PENDING → PROCESSING → COMPLETE / FAILED`
- The SQL above runs as a Databricks SQL batch job — `ai_query()` parallelizes across serverless SQL clusters automatically
- For throughput control: partition the batch by date/rack and run multiple concurrent SQL statements
- Auto Loader can feed new SVS arrivals into `svs_metadata` as `PENDING`, triggering incremental VLM runs via a scheduled Lakeflow Job



# Code

## Data Flow Diagram

```mermaid
flowchart TD
    %% ─── External Sources ───
    SVS_INPUT[("/Volumes/hls_pathology/orthanc_demo/raw_images/Aperio/\n~14 SVS files")]
    VLM_EP{{"databricks-llama-4-maverick\n(VLM Endpoint)"}}

    %% ─── Delta Tables ───
    OBJ_CAT[("douglas_moore.pathology\n.object_catalog")]
    OBJ_RED[("douglas_moore.pathology\n.object_catalog_redaction")]
    TIFF_STG[("douglas_moore.pathology\n.tiff_results_staging")]

    %% ─── Volumes (File Storage) ───
    LABEL_VOL[("/Volumes/.../label_images/\nPNG sub-images")]
    TIFF_VOL[("/Volumes/.../tiff_deidentified/\nBigTIFF output")]
    TMP["/tmp/ (executor local)\nBigTIFF staging"]

    %% ─── Processing Steps ───
    DISCOVER["Cell 17: File Discovery\nSVSCatalog.catalog()"]
    META["Cell 18: Metadata Extraction\nSVSMetaExtractor (mapInPandas)\nOpenSlide properties → VARIANT"]
    PHI_TAGS["Cell 19: PHI Tag Report\n(display only)"]
    EXTRACT["Cell 20: Extract Sub-images\npandas_udf + OpenSlide\nassociated_images → PNG"]
    VLM_DETECT["Cell 22: VLM PHI Detection\nai_query(files => content)\nREAD_FILES + INSERT"]
    BUILD_DF["Cell 24: Build redaction_df\nJOIN catalog + redaction\nWHERE status = PENDING"]
    UDF["Cell 25-27: De-identify UDF\nmapInPandas + ThreadPoolExecutor\nredact_image + write_pyramidal_bigtiff"]
    MERGE["Cell 28: MERGE Results\nUPDATE status, paths, errors"]
    AUDIT["Cell 29: Audit Summary\n(display only)"]

    %% ─── Data Flows ───
    SVS_INPUT -->|"list files"| DISCOVER
    DISCOVER -->|"files_df (in-memory)"| META
    SVS_INPUT -->|"read OpenSlide props"| META
    META -->|"mode=append"| OBJ_CAT

    OBJ_CAT -->|"read meta:phi_tag_report"| PHI_TAGS

    SVS_INPUT -->|"read associated_images"| EXTRACT
    EXTRACT -->|"save PNG (sequential write)"| LABEL_VOL

    LABEL_VOL -->|"READ_FILES(binaryFile)"| VLM_DETECT
    VLM_DETECT -->|"ai_query()"| VLM_EP
    VLM_EP -->|"JSON response"| VLM_DETECT
    OBJ_CAT -->|"JOIN for path"| VLM_DETECT
    VLM_DETECT -->|"INSERT INTO"| OBJ_RED

    OBJ_CAT -->|"JOIN"| BUILD_DF
    OBJ_RED -->|"WHERE PENDING"| BUILD_DF

    BUILD_DF -->|"redaction_df"| UDF
    SVS_INPUT -->|"read tiles (OpenSlide)"| UDF
    UDF -->|"redacted PNGs (seq write)"| LABEL_VOL
    UDF -->|"write BigTIFF (seek+write)"| TMP
    TMP -->|"shutil.copy2 (seq write)"| TIFF_VOL
    UDF -->|"saveAsTable"| TIFF_STG

    TIFF_STG -->|"source for MERGE"| MERGE
    MERGE -->|"UPDATE status/paths"| OBJ_RED

    OBJ_CAT -->|"LEFT JOIN"| AUDIT
    OBJ_RED -->|"LEFT JOIN"| AUDIT

    %% ─── Styling ───
    classDef volume fill:#e8f5e9,stroke:#2e7d32
    classDef table fill:#e3f2fd,stroke:#1565c0
    classDef process fill:#fff3e0,stroke:#e65100
    classDef external fill:#fce4ec,stroke:#c62828
    classDef tmp fill:#f5f5f5,stroke:#616161,stroke-dasharray:5

    class SVS_INPUT,LABEL_VOL,TIFF_VOL volume
    class OBJ_CAT,OBJ_RED,TIFF_STG table
    class DISCOVER,META,PHI_TAGS,EXTRACT,VLM_DETECT,BUILD_DF,UDF,MERGE,AUDIT process
    class VLM_EP external
    class TMP tmp
```

### Legend
| Color | Meaning |
|---|---|
| Green | UC Volumes (file storage) |
| Blue | Delta Tables (Unity Catalog) |
| Orange | Processing steps (notebook cells) |
| Pink | External service (model endpoint) |
| Dashed gray | Ephemeral local storage (/tmp) |

### Key Write Patterns
| Target | Write Mode | Reason |
|---|---|---|
| `object_catalog` | `mode=append` | Idempotent cataloguing; dedup via path |
| `object_catalog_redaction` | `INSERT INTO` | One row per VLM detection run |
| `tiff_results_staging` | `mode=overwrite` | Ephemeral staging; replaced each run |
| `object_catalog_redaction` | `MERGE ... WHEN MATCHED UPDATE` | Update status after TIFF write |
| Label PNGs (volume) | Sequential FUSE write | PIL `img.save()` — no seek needed |
| BigTIFFs (volume) | `/tmp/` → `shutil.copy2` | tifffile needs seek; Volume FUSE does not support seek+write |

In [0]:
# Install core dependencies.
# databricks-pixels provides Catalog + Transformer base classes.
# openslide-bin bundles libopenslide.so so OpenSlide works on Serverless.
%pip install openslide-python openslide-bin tifffile imagecodecs Pillow easyocr -q

In [0]:
# The full pixels source tree (including svs/) lives in the workspace.
# Add the src directory to sys.path so `dbx.pixels` and `dbx.pixels.svs`
# are importable without a separate pip install.
import sys
import types
import importlib

_SRC_ROOT = "/Workspace/Users/douglas.moore@databricks.com/pixels-svs/src"
if _SRC_ROOT not in sys.path:
    sys.path.insert(0, _SRC_ROOT)
importlib.invalidate_caches()

# Load deidentify module (file is clean — no truncation needed)
_deident_path = f"{_SRC_ROOT}/dbx/pixels/svs/deidentify.py"
with open(_deident_path, "r") as _f:
    _clean_src = _f.read()
_deident_mod = types.ModuleType("dbx.pixels.svs.deidentify")
_deident_mod.__file__ = _deident_path
exec(compile(_clean_src, _deident_path, "exec"), _deident_mod.__dict__)
sys.modules["dbx.pixels.svs.deidentify"] = _deident_mod

from dbx.pixels.svs import SVSCatalog, SVSMetaExtractor, SVSTiffWriter
from dbx.pixels.svs.phi_tags import classify_tags, scrub_image_description

# ── Pipeline configuration ────────────────────────────────────────────────────
CATALOG      = "douglas_moore"
SCHEMA       = "pathology"
UC_TABLE     = f"{CATALOG}.{SCHEMA}.object_catalog"
UC_VOLUME    = f"{CATALOG}.{SCHEMA}.pixels_volume"
INPUT_PATH   = "/Volumes/hls_pathology/orthanc_demo/raw_images/Aperio/"
TIFF_VOLUME  = f"/Volumes/{CATALOG}/{SCHEMA}/tiff_deidentified"
LABEL_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/label_images"
VLM_ENDPOINT = "databricks-llama-4-maverick"

print(f"Input : {INPUT_PATH}")
print(f"Table : {UC_TABLE}")
print(f"TIFFs : {TIFF_VOLUME}")
print(f"Labels: {LABEL_VOLUME}")
print(f"VLM   : {VLM_ENDPOINT}")


In [0]:
%sql
-- Reset pipeline state for a clean end-to-end run.
-- Truncates data only; table structure and permissions preserved.
TRUNCATE TABLE douglas_moore.pathology.object_catalog;
TRUNCATE TABLE douglas_moore.pathology.object_catalog_redaction;

In [0]:
# Create schema and volumes (idempotent — safe to re-run)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.pixels_volume")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.tiff_deidentified")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.label_images")

# Initialise Delta tables:
#   object_catalog           — base DDL from databricks-pixels (unchanged)
#   object_catalog_redaction — unified SVS/DICOM DDL from CREATE_SVS_CATALOG.sql
catalog = SVSCatalog(spark, table=UC_TABLE, volume=UC_VOLUME)
catalog.init_tables()

print("Schema, volumes, and tables initialised.")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))


In [0]:
# Discover all SVS files under INPUT_PATH and register them in object_catalog.
# SVSCatalog.catalog() defaults pattern='*.svs'; also picks up sidecar .txt files.
files_df = catalog.catalog(INPUT_PATH)
print(f"Discovered {files_df.count()} files")
display(files_df.select("path", "local_path", "length", "modificationTime", "extension"))


In [0]:
# SVSMetaExtractor reads every SVS via OpenSlide (ThreadPoolExecutor, 32 concurrent).
# All properties + derived fields (width, height, levels, phi_tag_report) are merged
# into one JSON dict → parse_json() → VARIANT.  No schema changes to object_catalog.
extractor = SVSMetaExtractor(catalog, inputCol="local_path", outputCol="meta")
meta_df   = extractor._transform(files_df)

(
    meta_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(UC_TABLE)
)

print(f"Wrote {spark.table(UC_TABLE).count()} rows to {UC_TABLE}")

display(spark.sql(f"""
SELECT
  regexp_extract(path, '[^/]+$', 0)   AS filename,
  meta:width::int                      AS width,
  meta:height::int                     AS height,
  meta:level_count::int                AS levels,
  meta:has_label_image::boolean        AS has_label,
  meta:has_macro_image::boolean        AS has_macro,
  meta:`aperio.AppMag`::string         AS app_mag,
  meta:`aperio.MPP`::string            AS mpp,
  meta
FROM {UC_TABLE}
ORDER BY filename
"""))


In [0]:
# PHI / QUESTIONABLE tag values for every slide.
# Unpack the phi_tag_report VARIANT array stored in meta.
from pyspark.sql.functions import regexp_extract, col, explode, from_json, expr
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

phi_schema = ArrayType(StructType([
    StructField("tag", StringType()),
    StructField("value", StringType()),
    StructField("classification", StringType()),
]))

phi_df = (
    spark.table(UC_TABLE)
    .filter(expr("meta:phi_tag_report IS NOT NULL"))
    .withColumn("phi_tag_report_str", expr("cast(meta:phi_tag_report AS STRING)"))
    .withColumn("tags", from_json("phi_tag_report_str", phi_schema))
    .withColumn("elem", explode("tags"))
    .select(
        regexp_extract("path", r"[^/]+$", 0).alias("filename"),
        col("elem.tag").alias("tag"),
        col("elem.value").alias("value"),
        col("elem.classification").alias("classification"),
    )
    .filter(col("classification").isin("PHI", "QUESTIONABLE"))
    .orderBy("filename", "classification", "tag")
)
print(f"PHI/QUESTIONABLE findings: {phi_df.count()}")
display(phi_df)

In [0]:
# Extract label and macro sub-images from each SVS and save as PNGs.
# These are later submitted to the VLM (Cell 9) and used as audit artefacts.
# Uses a pandas_udf so extraction runs distributed across workers.
from pyspark.sql.functions import pandas_udf, regexp_extract, col
import pandas as pd
from pyspark.sql.types import StringType

_LABEL_VOL = LABEL_VOLUME  # captured in closure; serialised with the UDF

@pandas_udf(StringType())
def extract_subimages_udf(paths: pd.Series, stems: pd.Series) -> pd.Series:
    import openslide, os
    results = []
    for path, stem in zip(paths, stems):
        try:
            slide = openslide.OpenSlide(path)
            saved = []
            for name in ("label", "macro"):
                if name in slide.associated_images:
                    img = slide.associated_images[name].convert("RGB")
                    out = f"{_LABEL_VOL}/{stem}_{name}.png"
                    os.makedirs(os.path.dirname(out), exist_ok=True)
                    img.save(out)
                    saved.append(out)
            slide.close()
            results.append(",".join(saved))
        except Exception as e:
            results.append(f"ERROR: {e}")
    return pd.Series(results)

catalog_df = (
    spark.table(UC_TABLE)
    .withColumn("stem", regexp_extract(col("path"), r"([^/]+)\.svs$", 1))
)

extracted_df = catalog_df.withColumn(
    "extracted_images",
    extract_subimages_udf(col("local_path"), col("stem")),
)

display(extracted_df.select("path", "stem", "extracted_images"))
print(f"Label/macro PNGs written to {LABEL_VOLUME}")


In [0]:
# Display first 10 label sub-images extracted from SVS pathology slides
import os
from PIL import Image
import matplotlib.pyplot as plt

label_dir = LABEL_VOLUME
label_files = sorted([f for f in os.listdir(label_dir) if f.endswith("_label.png")])[:10]

ncols = min(5, len(label_files))
nrows = (len(label_files) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 5 * nrows))
if len(label_files) == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for i, fname in enumerate(label_files):
    img = Image.open(os.path.join(label_dir, fname))
    axes[i].imshow(img)
    axes[i].set_title(fname.replace("_label.png", ""), fontsize=9)
    axes[i].axis("off")

for j in range(len(label_files), len(axes)):
    axes[j].axis("off")

plt.suptitle("SVS Label Sub-Images (PHI candidates for VLM redaction)", fontsize=13)
plt.tight_layout()
plt.show()

In [0]:
%sql
-- Run VLM PHI detection on all label PNGs and insert results into object_catalog_redaction.
-- ai_query() submits the image binary directly via `files => content` (no base64 needed).
-- responseFormat => 'json_object' guarantees machine-parseable output.
INSERT INTO douglas_moore.pathology.object_catalog_redaction (
  redaction_id, path, extension, modality,
  has_phi, phi_elements, vlm_raw_response, model_endpoint,
  output_file_paths, label_image_path, macro_image_path,
  status, insert_timestamp, created_by
)
WITH vlm_raw AS (
  SELECT
    regexp_replace(f._metadata.file_name, '_label\.png$', '') AS stem,
    f._metadata.file_path                                      AS label_image_path,
    ai_query(
      'databricks-llama-4-maverick',
      'You are a HIPAA-compliant PHI detection system.
Analyze this pathology slide label image and identify all Protected Health Information:
patient names, dates, MRNs, accession numbers, barcodes, or other identifying text.
Return ONLY a json object — no prose, no markdown fences.
Schema: {"has_phi": bool, "phi_elements": [{"type": "name|dob|mrn|accession|barcode|other", "value_hint": "<first 3 chars only>", "bbox": {"x": int, "y": int, "w": int, "h": int}, "subimage": "label"}]}
If no PHI found: {"has_phi": false, "phi_elements": []}',
      files => content
    ) AS vlm_raw_response
  FROM READ_FILES(
    '/Volumes/douglas_moore/pathology/label_images/',
    format          => 'binaryFile',
    fileNamePattern => '*_label.png'
  ) f
),
joined AS (
  SELECT
    v.stem,
    v.label_image_path,
    v.vlm_raw_response,
    m.path AS obj_path,
    concat('/Volumes/douglas_moore/pathology/label_images/', v.stem, '_macro.png') AS macro_image_path
  FROM vlm_raw v
  JOIN douglas_moore.pathology.object_catalog m
    ON regexp_extract(m.path, '([^/]+)\.svs$', 1) = v.stem
)
SELECT
  uuid()                                                                   AS redaction_id,
  obj_path                                                                  AS path,
  'svs'                                                                     AS extension,
  'WSI'                                                                     AS modality,
  try_cast(get_json_object(vlm_raw_response, '$.has_phi') AS BOOLEAN)       AS has_phi,
  parse_json(get_json_object(vlm_raw_response, '$.phi_elements'))           AS phi_elements,
  vlm_raw_response,
  'databricks-llama-4-maverick'                                             AS model_endpoint,
  array(CAST(NULL AS STRING))                                               AS output_file_paths,
  label_image_path,
  macro_image_path,
  'PENDING'                                                                 AS status,
  current_timestamp()                                                       AS insert_timestamp,
  current_user()                                                            AS created_by
FROM joined


In [0]:
%sql
select * from douglas_moore.pathology.object_catalog_redaction

In [0]:
# Reload modules to pick up streaming TIFF writer
import importlib, sys
for mod_name in list(sys.modules):
    if mod_name.startswith("dbx.pixels.svs"):
        del sys.modules[mod_name]

# Join PENDING redaction rows (phi_elements from VLM) with object_catalog (local_path),
# run de-identification and produce pyramidal BigTIFFs.
redaction_df = spark.sql(f"""
SELECT
  o.local_path,
  o.path,
  r.redaction_id,
  to_json(r.phi_elements)  AS phi_elements_json
FROM {CATALOG}.{SCHEMA}.object_catalog            o
JOIN {CATALOG}.{SCHEMA}.object_catalog_redaction  r
  ON o.path = r.path
WHERE r.status = 'PENDING'
""")
print(f"Files to de-identify: {redaction_df.count()}")

In [0]:
# --- Distributed de-identification via mapInPandas (memory-safe for Serverless 1 GB) ---
#
# Design principles applied from review:
# • No inner ThreadPoolExecutor — mapInPandas already parallelizes across Spark
#   partitions; nested threading doubles slide opens and memory pressure.
# • No to_dict("records") — iterate rows via iloc to avoid duplicating the batch.
# • OpenSlide closed in a finally block so C-bindings are destroyed even on error.
# • gc.collect() after each slide reclaims PIL/OpenSlide C-level allocations.
# • Tile-based TIFF writing via write_pyramidal_bigtiff_streaming (256×256 read_region).
# • Temp files staged to /tmp (seek-capable), then shutil.copy2 to Volume (seq FUSE).
# • PID suffix on temp paths prevents collisions across retries/speculative tasks.
# • repartition(num_slides) ensures 1 row per partition — each executor handles
#   exactly one slide. (arrow.maxRecordsPerBatch is NOT settable on Serverless.)

import json
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, IntegerType

_TIFF_VOLUME = TIFF_VOLUME
_LABEL_VOLUME = LABEL_VOLUME
_SRC_ROOT = "/Workspace/Users/douglas.moore@databricks.com/pixels-svs/src"

_result_schema = StructType([
    StructField("path", StringType(), True),
    StructField("tiff_output_path", StringType(), True),
    StructField("label_image_path", StringType(), True),
    StructField("macro_image_path", StringType(), True),
    StructField("phi_tags_redacted", ArrayType(StringType()), True),
    StructField("pixel_regions_redacted", IntegerType(), True),
    StructField("error", StringType(), True),
])


def _deidentify_batch(iterator):
    """mapInPandas worker: one slide per batch, streaming tile reads, no threading."""
    import sys, os, gc, shutil, time, logging, resource
    from pathlib import Path

    # --- Memory debugging utilities ---
    logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
    log = logging.getLogger("deidentify_worker")

    def _mem_mb() -> dict:
        """Return RSS and VMS in MB from /proc/self/status (Linux) with fallback."""
        rss_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024  # KB→MB on Linux
        try:
            with open("/proc/self/status") as f:
                status = f.read()
            vmpeak = vmrss = vmsize = 0
            for line in status.splitlines():
                if line.startswith("VmPeak:"):
                    vmpeak = int(line.split()[1]) / 1024
                elif line.startswith("VmRSS:"):
                    vmrss = int(line.split()[1]) / 1024
                elif line.startswith("VmSize:"):
                    vmsize = int(line.split()[1]) / 1024
            return {"rss_mb": round(vmrss, 1), "vms_mb": round(vmsize, 1), "peak_mb": round(vmpeak, 1)}
        except Exception:
            return {"rss_mb": round(rss_mb, 1), "vms_mb": -1, "peak_mb": -1}

    def _log_mem(stage: str, stem: str, extra: str = ""):
        mem = _mem_mb()
        msg = f"[{stem}] stage={stage} | RSS={mem['rss_mb']}MB VMS={mem['vms_mb']}MB Peak={mem['peak_mb']}MB"
        if extra:
            msg += f" | {extra}"
        log.info(msg)
        # Warn if approaching the 1024 MB limit
        if mem["rss_mb"] > 800:
            log.warning(f"⚠️  HIGH MEMORY [{stem}] stage={stage} RSS={mem['rss_mb']}MB — approaching 1024MB limit!")

    # Ensure src modules are importable on executors
    if _SRC_ROOT not in sys.path:
        sys.path.insert(0, _SRC_ROOT)

    import openslide
    from dbx.pixels.svs.deidentify import redact_image, write_pyramidal_bigtiff_streaming
    from dbx.pixels.svs.phi_tags import scrub_image_description

    for pdf in iterator:
        results = []
        log.info(f"Batch received: {len(pdf)} row(s) | PID={os.getpid()}")
        _log_mem("batch_start", "batch", f"rows={len(pdf)}")

        # Iterate rows directly via iloc — no to_dict("records") memory copy
        for idx in range(len(pdf)):
            row = pdf.iloc[idx]
            svs_path = row["local_path"]
            phi_json = row["phi_elements_json"]
            stem = Path(svs_path).stem
            stage = "init"
            slide = None
            tmp_tiff = f"/tmp/{stem}_{os.getpid()}.tiff"
            t0 = time.time()

            log.info(f"=== Processing slide: {stem} ===")
            _log_mem("init", stem, f"svs_path={svs_path}")

            try:
                phi_elements = json.loads(phi_json) if phi_json else []
                stage = "open_slide"
                slide = openslide.OpenSlide(svs_path)
                dims = slide.dimensions  # (width, height) at level 0
                levels = slide.level_count
                _log_mem("open_slide", stem, f"dims={dims[0]}x{dims[1]} levels={levels}")

                # 1. Redact label/macro sub-images (small RGBA → write PNGs to Volume)
                label_path = macro_path = None
                pixel_count = 0

                if "label" in slide.associated_images:
                    stage = "redact_label"
                    label_img = slide.associated_images["label"]
                    label_phi = [e for e in phi_elements if e.get("subimage", "label") != "macro"]
                    label_out = redact_image(label_img, label_phi)
                    pixel_count += len(label_phi)
                    label_path = f"{_LABEL_VOLUME}/{stem}_label.png"
                    os.makedirs(os.path.dirname(label_path), exist_ok=True)
                    label_out.save(label_path)
                    _log_mem("redact_label", stem, f"label_size={label_img.size} phi_count={len(label_phi)}")
                    del label_img, label_out  # free RGBA buffer immediately

                if "macro" in slide.associated_images:
                    stage = "redact_macro"
                    macro_img = slide.associated_images["macro"]
                    macro_phi = [e for e in phi_elements if e.get("subimage") == "macro"]
                    macro_out = redact_image(macro_img, macro_phi)
                    pixel_count += len(macro_phi)
                    macro_path = f"{_LABEL_VOLUME}/{stem}_macro.png"
                    os.makedirs(os.path.dirname(macro_path), exist_ok=True)
                    macro_out.save(macro_path)
                    _log_mem("redact_macro", stem, f"macro_size={macro_img.size} phi_count={len(macro_phi)}")
                    del macro_img, macro_out

                # 2. Scrub metadata tags
                stage = "scrub_metadata"
                raw_desc = slide.properties.get("tiff.ImageDescription", "")
                scrubbed = scrub_image_description(raw_desc)
                phi_tags_redacted = (
                    ["tiff.ImageDescription", "openslide.comment"]
                    if raw_desc != scrubbed else []
                )
                _log_mem("scrub_metadata", stem)

                # 3. Write pyramidal BigTIFF — tile-streaming (256×256 read_region)
                #    Stage to /tmp (requires seek), then sequential copy to Volume.
                stage = "write_tiff_to_tmp"
                t_tiff_start = time.time()
                write_pyramidal_bigtiff_streaming(tmp_tiff, slide, jpeg_quality=80)
                t_tiff_elapsed = time.time() - t_tiff_start
                tmp_size_mb = os.path.getsize(tmp_tiff) / (1024 * 1024) if os.path.exists(tmp_tiff) else 0
                _log_mem("write_tiff_done", stem, f"tiff_size={tmp_size_mb:.1f}MB elapsed={t_tiff_elapsed:.1f}s")

                # Close slide BEFORE copy to free C-level handles and mapped memory
                slide.close()
                slide = None
                _log_mem("slide_closed", stem)

                stage = "copy_tiff_to_volume"
                t_copy_start = time.time()
                tiff_path = f"{_TIFF_VOLUME}/{stem}.tiff"
                shutil.copy2(tmp_tiff, tiff_path)
                os.remove(tmp_tiff)
                t_copy_elapsed = time.time() - t_copy_start
                _log_mem("copy_done", stem, f"copy_elapsed={t_copy_elapsed:.1f}s")

                total_elapsed = time.time() - t0
                log.info(f"✓ [{stem}] completed in {total_elapsed:.1f}s | tiff={tmp_size_mb:.1f}MB")

                results.append({
                    "path": row["path"],
                    "tiff_output_path": tiff_path,
                    "label_image_path": label_path,
                    "macro_image_path": macro_path,
                    "phi_tags_redacted": phi_tags_redacted,
                    "pixel_regions_redacted": pixel_count,
                    "error": None,
                })

            except Exception as exc:
                _log_mem("ERROR", stem, f"stage={stage} exc={type(exc).__name__}: {exc}")
                results.append({
                    "path": row["path"],
                    "tiff_output_path": None,
                    "label_image_path": None,
                    "macro_image_path": None,
                    "phi_tags_redacted": [],
                    "pixel_regions_redacted": 0,
                    "error": f"[stage={stage}] {type(exc).__name__}: {exc}",
                })

            finally:
                # Ensure slide is always closed — destroy C-bindings immediately
                if slide is not None:
                    try:
                        slide.close()
                    except Exception:
                        pass
                # Clean up temp file on failure
                if os.path.exists(tmp_tiff):
                    try:
                        os.remove(tmp_tiff)
                    except Exception:
                        pass
                # Force GC to reclaim PIL/OpenSlide C-level allocations
                gc.collect()
                _log_mem("gc_complete", stem)

        yield pd.DataFrame(results)

In [0]:
# Materialize the expensive mapInPandas UDF exactly ONCE.
# Strategy: persist() + count() forces a single execution pass.
# Downstream MERGE reads from the cached DataFrame via temp view.
TIFF_RESULTS_TABLE = f"{CATALOG}.{SCHEMA}.tiff_results_staging"

num_slides = redaction_df.count()
print(f"""{num_slides}""")
assert num_slides > 0, "No PENDING slides to de-identify — check object_catalog_redaction status"


In [0]:
# --- Dry run: 1 slide on DRIVER to capture full memory trace ---
# Runs the same logic outside mapInPandas so we can see exactly which stage
# exceeds 1024 MB without the executor being killed.

import sys, os, gc, time, resource, json, shutil
from pathlib import Path

if "/Workspace/Users/douglas.moore@databricks.com/pixels-svs/src" not in sys.path:
    sys.path.insert(0, "/Workspace/Users/douglas.moore@databricks.com/pixels-svs/src")

import openslide
from dbx.pixels.svs.deidentify import redact_image, write_pyramidal_bigtiff_streaming
from dbx.pixels.svs.phi_tags import scrub_image_description

def _mem_mb():
    """RSS/VMS/Peak from /proc/self/status."""
    try:
        with open("/proc/self/status") as f:
            status = f.read()
        vals = {}
        for line in status.splitlines():
            for key in ("VmPeak", "VmRSS", "VmSize"):
                if line.startswith(key + ":"):
                    vals[key] = int(line.split()[1]) / 1024  # KB→MB
        return {"rss_mb": round(vals.get("VmRSS", 0), 1),
                "vms_mb": round(vals.get("VmSize", 0), 1),
                "peak_mb": round(vals.get("VmPeak", 0), 1)}
    except Exception:
        return {"rss_mb": -1, "vms_mb": -1, "peak_mb": -1}

def log_mem(stage, extra=""):
    mem = _mem_mb()
    warn = " ⚠️ OVER 1GB!" if mem["rss_mb"] > 1024 else ("⚠️ HIGH" if mem["rss_mb"] > 800 else "")
    print(f"  [{stage:20s}] RSS={mem['rss_mb']:>7.1f}MB  VMS={mem['vms_mb']:>7.1f}MB  Peak={mem['peak_mb']:>7.1f}MB {warn}  {extra}")

# Get one slide from the redaction dataframe
row = redaction_df.limit(1).collect()[0]
svs_path = row["local_path"]
phi_json = row["phi_elements_json"]
stem = Path(svs_path).stem
tmp_tiff = f"/tmp/{stem}_dryrun.tiff"

print(f"\n{'='*80}")
print(f"DRY RUN MEMORY PROFILE: {stem}")
print(f"SVS path: {svs_path}")
print(f"{'='*80}")

gc.collect()
log_mem("baseline")

# Open slide
t0 = time.time()
slide = openslide.OpenSlide(svs_path)
dims = slide.dimensions
print(f"\n  Slide: {dims[0]}x{dims[1]} pixels, {slide.level_count} levels")
print(f"  Level dimensions: {[slide.level_dimensions[i] for i in range(slide.level_count)]}")
print(f"  Associated images: {list(slide.associated_images.keys())}")
log_mem("open_slide", f"file_size={os.path.getsize(svs_path)/(1024*1024):.1f}MB")

# Redact label
if "label" in slide.associated_images:
    label_img = slide.associated_images["label"]
    print(f"\n  Label image: {label_img.size} mode={label_img.mode}")
    phi_elements = json.loads(phi_json) if phi_json else []
    label_phi = [e for e in phi_elements if e.get("subimage", "label") != "macro"]
    label_out = redact_image(label_img, label_phi)
    log_mem("redact_label", f"phi_regions={len(label_phi)}")
    del label_img, label_out
    gc.collect()
    log_mem("label_freed")

# Redact macro
if "macro" in slide.associated_images:
    macro_img = slide.associated_images["macro"]
    print(f"\n  Macro image: {macro_img.size} mode={macro_img.mode}")
    phi_elements = json.loads(phi_json) if phi_json else []
    macro_phi = [e for e in phi_elements if e.get("subimage") == "macro"]
    macro_out = redact_image(macro_img, macro_phi)
    log_mem("redact_macro", f"phi_regions={len(macro_phi)}")
    del macro_img, macro_out
    gc.collect()
    log_mem("macro_freed")

# Scrub metadata
raw_desc = slide.properties.get("tiff.ImageDescription", "")
scrubbed = scrub_image_description(raw_desc)
log_mem("scrub_metadata")

# Write pyramidal BigTIFF (this is the suspected memory hog)
print(f"\n  Writing pyramidal BigTIFF to /tmp ...")
t_tiff = time.time()
try:
    write_pyramidal_bigtiff_streaming(tmp_tiff, slide, jpeg_quality=80)
    tiff_elapsed = time.time() - t_tiff
    tiff_size = os.path.getsize(tmp_tiff) / (1024 * 1024)
    log_mem("write_tiff_done", f"size={tiff_size:.1f}MB elapsed={tiff_elapsed:.1f}s")
except Exception as e:
    log_mem("write_tiff_FAILED", f"{type(e).__name__}: {e}")
    tiff_size = 0

# Close slide
slide.close()
log_mem("slide_closed")
gc.collect()
log_mem("gc_after_close")

# Cleanup
if os.path.exists(tmp_tiff):
    os.remove(tmp_tiff)

total = time.time() - t0
print(f"\n{'='*80}")
print(f"COMPLETE: {stem} in {total:.1f}s | TIFF={tiff_size:.1f}MB")
print(f"Peak memory: {_mem_mb()['peak_mb']:.1f}MB")
print(f"{'='*80}")
if _mem_mb()["peak_mb"] > 1024:
    print("\n❌ Peak memory EXCEEDED 1024MB — this will OOM on Serverless executors.")
    print("   → Investigate write_pyramidal_bigtiff_streaming tile buffer size.")
else:
    print("\n✅ Peak memory stayed under 1024MB — safe for Serverless executors.")

In [0]:

# 1 slide per partition to stay under 1GB UDF memory limit on serverless.
# Large SVS files (CMU-1 = 46000x32000) need sole access to executor RAM.
results_df = (
    redaction_df
    .repartition(num_slides)
    .mapInPandas(_deidentify_batch, schema=_result_schema)
    .limit(4)
)

# Force single execution — UDF runs here and only here
results_df.select("path", "tiff_output_path", "label_image_path", "macro_image_path", "phi_tags_redacted", "pixel_regions_redacted", "error").write.saveAsTable(TIFF_RESULTS_TABLE)


display(spark.sql(f"SELECT * FROM {TIFF_RESULTS_TABLE}"))

In [0]:
# Merge output paths and status back into object_catalog_redaction
spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.object_catalog_redaction AS tgt
USING (
  SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY path ORDER BY path) AS rn
    FROM tiff_results
  ) WHERE rn = 1
) AS src
  ON tgt.path = src.path
WHEN MATCHED THEN UPDATE SET
  tgt.output_file_paths      = array(src.tiff_output_path),
  tgt.label_image_path       = COALESCE(src.label_image_path, tgt.label_image_path),
  tgt.macro_image_path       = COALESCE(src.macro_image_path, tgt.macro_image_path),
  tgt.phi_tags_redacted      = src.phi_tags_redacted,
  tgt.pixel_redactions_count = src.pixel_regions_redacted,
  tgt.status                 = CASE WHEN src.error IS NULL THEN 'SUCCESS' ELSE 'FAILED' END,
  tgt.error_messages         = CASE WHEN src.error IS NOT NULL THEN array(src.error) ELSE NULL END,
  tgt.update_timestamp       = current_timestamp()
""")
print("TIFF write complete. Redaction records updated.")

In [0]:
# End-to-end audit: join object_catalog with object_catalog_redaction and summarise.
audit_df = spark.sql(f"""
SELECT
  regexp_extract(o.path, '[^/]+$', 0)    AS filename,
  o.meta:width::int                       AS width_px,
  o.meta:height::int                      AS height_px,
  o.meta:level_count::int                 AS pyramid_levels,
  r.has_phi,
  r.status,
  r.pixel_redactions_count,
  size(r.phi_tags_redacted)               AS tag_redactions,
  r.output_file_paths[0]                  AS tiff_output_path,
  r.label_image_path,
  r.error_messages[0]                     AS error
FROM {CATALOG}.{SCHEMA}.object_catalog            o
LEFT JOIN {CATALOG}.{SCHEMA}.object_catalog_redaction r
  ON o.path = r.path
ORDER BY filename
""")

total   = audit_df.count()
phi_ct  = audit_df.filter("has_phi = true").count()
ok_ct   = audit_df.filter("status = 'SUCCESS'").count()
err_ct  = audit_df.filter("status = 'FAILED'").count()

print(f"Slides in catalog       : {total}")
print(f"VLM-flagged with PHI    : {phi_ct}")
print(f"Successfully written    : {ok_ct}")
print(f"Errors                  : {err_ct}")

display(audit_df)
